### Group 24

Shiref Khaled Elhalawany -  221100944

Ahmed Anis Hassan - 221100101 

Karim Ashraf Elsayed - 221100391

Kareem Shaheen - 221101524

# Part 1: PCA Method with Mean-Filling

### Importing Libraries and Defining Paths

This code imports the required Python libraries for data processing and visualization, including Pandas, NumPy, and Matplotlib. It then defines file paths for the scaled ratings dataset and the selected users and items tables. The code ensures that the output directory exists before proceeding and prints a confirmation message indicating that the environment is properly set up for the next processing steps.

In [140]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

tables_path = '../results/tables/'
scaled_ratings_path = tables_path + 'sampled_scaled_ratings.csv'
selected_items_path = tables_path + '3_1_12_selected_items.csv'
selected_users_path = tables_path + '3_1_11_selected_users.csv'

if not os.path.exists(tables_path):
    os.makedirs(tables_path)

print("Libraries imported and paths defined.")


runtime_rows = []

def log_time(method, stage, seconds, extra=""):
    runtime_rows.append({
        "Method": method,
        "Stage": stage,
        "Seconds": seconds,
        "Extra": extra
    })


def memory_mb(*arrays):
    total_bytes = 0
    for arr in arrays:
        if hasattr(arr, "values"):      
            total_bytes += arr.values.nbytes
        else:                            
            total_bytes += arr.nbytes
    return total_bytes / (1024 ** 2)

Libraries imported and paths defined.


### Data Loading

In [141]:
df_ratings = pd.read_csv(scaled_ratings_path)
df_items = pd.read_csv(selected_items_path)
df_users = pd.read_csv(selected_users_path)

target_user_ids = df_users['UserID'].tolist()

print("Datasets loaded.")

top_n = 1000
item_counts = df_ratings['ItemID'].value_counts()
top_items = item_counts.head(top_n).index.tolist()

target_item_ids = df_items['ItemID'].tolist()
for tid in target_item_ids:
    if tid not in top_items:
        top_items.append(tid)

df_filtered = df_ratings[df_ratings['ItemID'].isin(top_items)].copy()

user_item_matrix = df_filtered.pivot(index='UserID', columns='ItemID', values='Rating')

print(f"User-Item Matrix Shape: {user_item_matrix.shape}")

Datasets loaded.
User-Item Matrix Shape: (28301, 500)


### Step 1: Computing Target Item Averages and Inspecting Missing Values

This code calculates the average rating for each target item using only the available user ratings from the user–item matrix. It prints the raw average values and saves them to a CSV file for later analysis. The code then displays a sample of the user–item matrix before any preprocessing is applied, highlighting the presence of missing (NaN) values. Finally, it counts and reports the total number of missing entries and saves the matrix with unspecified values to a CSV file for reference and documentation.

In [142]:
target_item_ids = df_items['ItemID'].tolist()
target_avgs = user_item_matrix[target_item_ids].mean()

print("Step 1: Average ratings for target items (Raw):")
print(target_avgs)

target_avgs.to_csv(tables_path + 'pca_step1_target_avgs.csv', header=['AverageRating'])
print("Step 1 Output Saved.")

Step 1: Average ratings for target items (Raw):
ItemID
99904     1.0
119705    1.0
dtype: float64
Step 1 Output Saved.


In [143]:
print("Matrix with Unspecified Values (NaN):")
print(user_item_matrix.head())
print(f"\nTotal Missing Values: {user_item_matrix.isnull().sum().sum()}")

save_path_matrix = tables_path + 'matrix_before_mean_filling.csv'
user_item_matrix.to_csv(save_path_matrix)
print(f"Saved matrix with NaNs to: {save_path_matrix}")

Matrix with Unspecified Values (NaN):
ItemID  34      93      171     179     222     231     287     289     \
UserID                                                                   
1          NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
6          NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
19         NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
26         NaN     NaN     NaN     NaN     NaN     2.0     NaN     NaN   
38         NaN     NaN     NaN     NaN     3.0     NaN     NaN     5.0   

ItemID  319     458     ...  129391  129405  129739  129771  130083  130298  \
UserID                  ...                                                   
1          NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
6          NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
19         NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
26         NaN     NaN  ...     NaN     NaN     

### Step 2: Mean-Filling Missing Ratings for PCA Preparation

This code applies the mean-filling technique to handle missing values in the user–item matrix. Each unspecified rating (NaN) is replaced with the average rating of the corresponding item, allowing the matrix to become fully dense. This preprocessing step is required to enable Principal Component Analysis (PCA), which cannot operate on matrices with missing values. The completed matrix is then saved to a CSV file, and a preview of the filled matrix is printed for verification.

In [144]:
user_item_matrix_filled = user_item_matrix.fillna(user_item_matrix.mean())

print("Step 2: Missing values filled with item means.")

user_item_matrix_filled.to_csv(tables_path + 'pca_step2_filled_matrix.csv')
print("Step 2 Output Saved.")
print(user_item_matrix_filled.head())

Step 2: Missing values filled with item means.
Step 2 Output Saved.
ItemID    34        93        171       179       222      231       287     \
UserID                                                                        
1       3.716688  2.583612  3.520095  2.783562  3.548643  3.05222  3.205882   
6       3.716688  2.583612  3.520095  2.783562  3.548643  3.05222  3.205882   
19      3.716688  2.583612  3.520095  2.783562  3.548643  3.05222  3.205882   
26      3.716688  2.583612  3.520095  2.783562  3.548643  2.00000  3.205882   
38      3.716688  2.583612  3.520095  2.783562  3.000000  3.05222  3.205882   

ItemID    289       319       458     ...  129391  129405  129739  129771  \
UserID                                ...                                   
1       3.351588  3.881146  3.139466  ...     3.0     2.5     5.0     4.0   
6       3.351588  3.881146  3.139466  ...     3.0     2.5     5.0     4.0   
19      3.351588  3.881146  3.139466  ...     3.0     2.5     5.0     

### Step 3: Computing Item Mean Ratings After Mean-Filling

This code calculates the average rating for each item using the fully mean-filled user–item matrix. These values represent the final item mean ratings and should match the means used during the missing-value filling process. The computed averages are printed for verification and saved to a CSV file, providing a reference for later PCA steps and result interpretation.

In [145]:
item_means = user_item_matrix_filled.mean()

print("Step 3: Average ratings calculated.")

item_means.to_csv(tables_path + 'pca_step3_item_means.csv', header=['MeanRating'])
print("Step 3 Output Saved.")
print(item_means.head())

Step 3: Average ratings calculated.
Step 3 Output Saved.
ItemID
34     3.716688
93     2.583612
171    3.520095
179    2.783562
222    3.548643
dtype: float64


### Step 4: Mean-Centering the User–Item Matrix

This code performs mean-centering on the user–item matrix by subtracting the average rating of each item from its corresponding ratings. As a result, each item’s ratings are transformed to represent deviations from the item mean rather than absolute rating values. Mean-centering is a critical preprocessing step for Principal Component Analysis (PCA), as it ensures that the extracted components capture variance patterns instead of baseline rating biases. The centered matrix is saved to a CSV file, and a preview is printed for verification.

In [146]:
user_item_matrix_centered = user_item_matrix_filled - item_means

print("Step 4: Data mean-centered.")

user_item_matrix_centered.to_csv(tables_path + 'pca_step4_centered_matrix.csv')
print("Step 4 Output Saved.")
print(user_item_matrix_centered.head())

Step 4: Data mean-centered.
Step 4 Output Saved.
ItemID        34      93            171           179       222     \
UserID                                                               
1       8.881784e-16     0.0 -8.881784e-16 -1.332268e-15  0.000000   
6       8.881784e-16     0.0 -8.881784e-16 -1.332268e-15  0.000000   
19      8.881784e-16     0.0 -8.881784e-16 -1.332268e-15  0.000000   
26      8.881784e-16     0.0 -8.881784e-16 -1.332268e-15  0.000000   
38      8.881784e-16     0.0 -8.881784e-16 -1.332268e-15 -0.548643   

ItemID        231           287           289           319           458     \
UserID                                                                         
1      -4.440892e-16  4.440892e-16  4.440892e-16 -4.440892e-16  1.332268e-15   
6      -4.440892e-16  4.440892e-16  4.440892e-16 -4.440892e-16  1.332268e-15   
19     -4.440892e-16  4.440892e-16  4.440892e-16 -4.440892e-16  1.332268e-15   
26     -1.052220e+00  4.440892e-16  4.440892e-16 -4.440892e-

### Step 5 & 6: Computing the Item–Item Covariance Matrix

This code computes the covariance matrix between items using the mean-centered user–item matrix. Since users are represented as rows and items as columns, the covariance is calculated across columns to capture how item ratings vary together across users. The resulting covariance matrix quantifies relationships between pairs of items and serves as a core input for Principal Component Analysis (PCA). For easier interpretation and indexing, the matrix is stored as a Pandas DataFrame, saved to a CSV file, and its dimensions and a sample of values are printed for verification.

In [147]:
t0 = time.perf_counter()

cov_matrix = np.cov(user_item_matrix_centered, rowvar=False)

cov_df = pd.DataFrame(cov_matrix, index=user_item_matrix_centered.columns, columns=user_item_matrix_centered.columns)

print("Step 6: Covariance Matrix Generated.")
print(cov_df.shape)

cov_df.to_csv(tables_path + 'pca_step6_cov_matrix.csv')
print("Step 6 Output Saved.")
print(cov_df.head())

t1 = time.perf_counter()
log_time("PCA_MeanFill", "Covariance_Estimation", t1 - t0, extra=f"k={k}")
print("PCA Mean Covariance_Estimation time:", t1 - t0)

Step 6: Covariance Matrix Generated.
(500, 500)
Step 6 Output Saved.
ItemID    34        93        171       179       222       231       287     \
ItemID                                                                         
34      0.370582  0.001334 -0.000245 -0.000629  0.004843  0.012846  0.000196   
93      0.001334  0.022591 -0.000097  0.001332  0.000568  0.003112 -0.000004   
171    -0.000245 -0.000097  0.015109  0.000816  0.001039  0.000913  0.000770   
179    -0.000629  0.001332  0.000816  0.014555  0.001449  0.002044  0.000691   
222     0.004843  0.000568  0.001039  0.001449  0.046990  0.000867  0.000913   

ItemID    289       319       458     ...  129391    129405  129739  129771  \
ItemID                                ...                                     
34      0.000485  0.002171  0.000061  ...     0.0  0.000144     0.0     0.0   
93      0.001531  0.000695  0.000665  ...     0.0  0.000000     0.0     0.0   
171     0.001007  0.001049  0.000442  ...     0.0  0.0

### Step 7: Identifying Item Peers Using PCA Latent Space

This code performs eigen-decomposition on the item–item covariance matrix to extract eigenvalues and eigenvectors, then sorts them in descending order to prioritize the most important principal components. Instead of choosing a fixed number of components, the code calculates the explained variance ratio for each component and uses the cumulative explained variance to automatically determine the smallest number of components needed to explain at least 75% of the total variance.

Using this dynamically selected value of k, the code constructs a latent item feature matrix (items × k) from the top eigenvectors. It then computes cosine similarity between each target item and all other items within this reduced latent space. For each target item, the code identifies and stores the Top-5 and Top-10 most similar peer items, prints them for inspection, and saves the final peer lists to a CSV file for later prediction and recommendation steps.

In [148]:
t0 = time.perf_counter()

eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

sorted_indices = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indices]
eigenvectors = eigenvectors[:, sorted_indices]

print("Eigen-decomposition complete.")

total_variance = np.sum(eigenvalues)
explained_variance_ratio = eigenvalues / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

k_75 = np.argmax(cumulative_variance >= 0.75) + 1
print(f"\nNumber of components to explain 75% variance: {k_75}")
print(f"Correlation Variance at k={k_75}: {cumulative_variance[k_75-1]:.4f}")

k = k_75

print(f"Selected Top {k} Eigenvalues based on 75% Variance:")
print(eigenvalues[:k])

t1 = time.perf_counter()
log_time("PCA_MeanFill", "EigenDecomposition", t1 - t0, extra=f"k={k}")
print("PCA Mean EigenDecomposition time:", t1 - t0)

latent_item_features_5 = eigenvectors[:, :k]
latent_item_df_5 = pd.DataFrame(latent_item_features_5, index=user_item_matrix_centered.columns)

W_k = eigenvectors[:, :k] 

R_centered = user_item_matrix_centered.values.astype(float)

_item_means_vec = user_item_matrix.mean(axis=0).values.astype(float) 

Z = R_centered @ W_k

R_hat_centered = Z @ W_k.T

R_hat = R_hat_centered + _item_means_vec

R_hat_df = pd.DataFrame(R_hat, index=user_item_matrix.index, columns=user_item_matrix.columns)
out_file = tables_path + f"pca_meanfill_k{k}_reconstruction.csv"
R_hat_df.to_csv(out_file)

print(f"PCA reconstructed matrix saved: {out_file}")

R_true = user_item_matrix_filled.values.astype(float)
R_pred = R_hat_df.values.astype(float)

R_true_flat = R_true.flatten()
R_pred_flat = R_pred.flatten()

mae = mean_absolute_error(R_true_flat, R_pred_flat)
rmse = np.sqrt(mean_squared_error(R_true_flat, R_pred_flat))

explained_variance = float(np.sum(eigenvalues[:k]) / np.sum(eigenvalues))

print(f"PCA MeanFill Reconstruction (k={k}) -> MAE={mae:.6f}, RMSE={rmse:.6f}, ExplainedVariance={explained_variance:.6f}")

pca_error_row = pd.DataFrame([{
    "k": k,
    "MAE": mae,
    "RMSE": rmse,
    "ExplainedVariance": explained_variance
}])

pca_err_file = tables_path + "pca_meanfill_reconstruction_errors.csv"

if os.path.exists(pca_err_file):
    old = pd.read_csv(pca_err_file)
    combined = pd.concat([old, pca_error_row], ignore_index=True)
    combined = combined.drop_duplicates(subset=["k"], keep="last").sort_values("k")
    combined.to_csv(pca_err_file, index=False)
else:
    pca_error_row.to_csv(pca_err_file, index=False)

print(f"Saved PCA mean-fill reconstruction errors to: {pca_err_file}")

Eigen-decomposition complete.

Number of components to explain 75% variance: 32
Correlation Variance at k=32: 0.7569
Selected Top 32 Eigenvalues based on 75% Variance:
[0.4839078  0.37386972 0.21383942 0.13397112 0.12283088 0.11012668
 0.09901522 0.0940033  0.08364928 0.08014133 0.07589395 0.07095707
 0.06663785 0.06448294 0.05843734 0.05451769 0.05117446 0.04837373
 0.04698586 0.04484818 0.04145906 0.03910595 0.03824191 0.03716566
 0.0343978  0.03339317 0.03253795 0.03201134 0.03175114 0.02848674
 0.02795814 0.0268384 ]
PCA Mean EigenDecomposition time: 0.3475208999989263
PCA reconstructed matrix saved: ../results/tables/pca_meanfill_k32_reconstruction.csv
PCA MeanFill Reconstruction (k=32) -> MAE=0.003741, RMSE=0.042270, ExplainedVariance=0.756861
Saved PCA mean-fill reconstruction errors to: ../results/tables/pca_meanfill_reconstruction_errors.csv


In [149]:
memory_pca_mean = memory_mb(
    cov_matrix,
    eigenvectors,
    R_hat_df
)

memory_df = pd.DataFrame([{
    "Method": "PCA_MeanFill",
    "Components": "Covariance + Eigenvectors + Reconstruction",
    "Memory_MB": memory_pca_mean
}])

memory_df.to_csv(
    tables_path + "pca_meanfill_memory.csv",
    index=False
)

print("Saved PCA Mean-Fill memory to pca_meanfill_memory.csv")
print(memory_df)

Saved PCA Mean-Fill memory to pca_meanfill_memory.csv
         Method                                  Components   Memory_MB
0  PCA_MeanFill  Covariance + Eigenvectors + Reconstruction  111.774445


In [150]:
peers_dict = {}
peers_data = []

for tid in target_item_ids:
    target_vec = latent_item_df_5.loc[tid].values.reshape(1, -1)

    sim_scores = cosine_similarity(target_vec, latent_item_df_5.values).flatten()
    sim_series = pd.Series(sim_scores, index=latent_item_df_5.index)

    sim_series = sim_series.drop(tid)
    sorted_peers = sim_series.sort_values(ascending=False)

    top5 = sorted_peers.head(5).index.tolist()
    top10_ids = sorted_peers.head(10).index.tolist()

    peers_dict[tid] = {
        'top5': top5,
        'top10': top10_ids,
        'sim_series_5': sim_series
    }

    peers_data.append({
    'TargetItem': tid,
    'Top5_Peers_Latent': str(top5),
    'Top10_Peers_Latent': str(top10_ids)})

    print(f"Item {tid} (Latent Space 75% Var):")
    print(f"  Top 5 Peers: {top5}")
    print(f"  Top 10 Peers: {top10_ids}")

pd.DataFrame(peers_data).to_csv(tables_path + 'pca_step7_latent_peers.csv', index=False)
print("Step 7 Output Saved.")

Item 99904 (Latent Space 75% Var):
  Top 5 Peers: [34, 92210, 94786, 94739, 94405]
  Top 10 Peers: [34, 92210, 94786, 94739, 94405, 93752, 93574, 93570, 93544, 93510]
Item 119705 (Latent Space 75% Var):
  Top 5 Peers: [34, 92210, 94786, 94739, 94405]
  Top 10 Peers: [34, 92210, 94786, 94739, 94405, 93752, 93574, 93570, 93544, 93510]
Step 7 Output Saved.


### Step 8: Constructing the Reduced Latent Space Using Top-5 Peers

This code builds a reduced-dimensional representation for each target item using its top 5 most similar peers identified in the PCA latent space. For every target item, the code stores the peer rank, peer item ID, and cosine similarity weight, creating a compact representation of the item’s local neighborhood in latent space. This reduced peer information is saved to a CSV file for later recommendation or prediction tasks.

As an additional step, the code constructs user-level reduced vectors by extracting each user’s mean-centered ratings for the top-5 peer items of every target item. These vectors represent user preferences within the reduced latent space and are saved separately. This step enables efficient similarity-based prediction and provides a transparent, evaluation-ready representation suitable for further modeling and analysis.

In [151]:
reduced_space_data = []

for tid in target_item_ids:
    top5_peers = peers_dict[tid]['top5']
    sim_series = peers_dict[tid]['sim_series_5']

    for rank, peer_id in enumerate(top5_peers, 1):
        weight = sim_series[peer_id]
        reduced_space_data.append({
            'TargetItem': tid,
            'Peer_Rank': rank,
            'Peer_ItemID': peer_id,
            'Latent_Similarity': weight,
            'Space_Type': 'Top5_Latent'
        })

df_reduced_5 = pd.DataFrame(reduced_space_data)
print(df_reduced_5.head())

df_reduced_5.to_csv(tables_path + 'pca_step8_reduced_space_top5.csv', index=False)
print("Step 8 Output Saved: pca_step8_reduced_space_top5.csv")

rows = []
for tid in target_item_ids:
    peers = peers_dict[tid]['top5']
    for uid in target_user_ids:
        if uid not in user_item_matrix_centered.index:
            continue

        vec = user_item_matrix_centered.loc[uid, peers].values 
        row = {'UserID': uid, 'TargetItem': tid}
        for i, p in enumerate(peers, 1):
            row[f'Peer{i}_{p}'] = vec[i-1]
        rows.append(row)

df_user_red5 = pd.DataFrame(rows)
print(df_user_red5.head())

df_user_red5.to_csv(tables_path + "pca_step8_user_reduced_vectors_top5.csv", index=False)
print("Step 8 (extra) Output Saved: pca_step8_user_reduced_vectors_top5.csv")

   TargetItem  Peer_Rank  Peer_ItemID  Latent_Similarity   Space_Type
0       99904          1           34                0.0  Top5_Latent
1       99904          2        92210                0.0  Top5_Latent
2       99904          3        94786                0.0  Top5_Latent
3       99904          4        94739                0.0  Top5_Latent
4       99904          5        94405                0.0  Top5_Latent
Step 8 Output Saved: pca_step8_reduced_space_top5.csv
   UserID  TargetItem      Peer1_34   Peer2_92210  Peer3_94786   Peer4_94739  \
0       1       99904  8.881784e-16  8.881784e-16          0.0  1.332268e-15   
1     134       99904  2.833118e-01  8.881784e-16          0.0  1.332268e-15   
2     903       99904  2.833118e-01  8.881784e-16          0.0  1.332268e-15   
3       1      119705  8.881784e-16  8.881784e-16          0.0  1.332268e-15   
4     134      119705  2.833118e-01  8.881784e-16          0.0  1.332268e-15   

   Peer5_94405  
0          0.0  
1          

### Step 9: Predicting Ratings Using Top-5 Latent Peers

This code generates rating predictions for each target user–item pair using the top 5 most similar items identified in the PCA latent space. For every user and target item, the code checks whether a rating already exists or is missing. It then applies a similarity-weighted prediction formula based on the user’s mean-centered ratings for the peer items and their corresponding latent-space similarity weights.

If no valid similarity weights are available, the prediction defaults to the item’s average rating. Otherwise, the weighted deviation is added back to the item mean to produce the final predicted rating. All predictions are stored along with their status (existing or missing) and saved to a CSV file for evaluation and analysis.

In [152]:
t0 = time.perf_counter()

predictions_top5 = []

for uid in target_user_ids:
    if uid not in user_item_matrix.index: continue
        
    for tid in target_item_ids:
        if pd.notna(user_item_matrix.loc[uid, tid]):
           status = "Existing"
        else:
           status = "Missing"
           
        current_peers = peers_dict[tid]['top5']
        sim_series = peers_dict[tid]['sim_series_5']
        
        numerator = 0
        denominator = 0
        
        for peer_id in current_peers:
            weight = sim_series[peer_id]
            val_centered = user_item_matrix_centered.loc[uid, peer_id]
            
            numerator += weight * val_centered
            denominator += abs(weight)
            
        if denominator == 0:
            pred = item_means[tid]
        else:
            pred = item_means[tid] + (numerator / denominator)
            
        predictions_top5.append({'UserID': uid, 'ItemID': tid, 'Pred_Top5': pred, 'Status': status})

df_pred5 = pd.DataFrame(predictions_top5)
print(df_pred5)
df_pred5.to_csv(tables_path + 'pca_step9_predictions_top5.csv', index=False)
print("Step 9 Output Saved.")

t1 = time.perf_counter()
log_time("PCA_MeanFill", "Prediction_Top5", t1 - t0)
print("PCA Mean Top-5 prediction time:", t1 - t0)

   UserID  ItemID  Pred_Top5   Status
0       1   99904        1.0  Missing
1       1  119705        1.0  Missing
2     134   99904        1.0  Missing
3     134  119705        1.0  Missing
4     903   99904        1.0  Missing
5     903  119705        1.0  Missing
Step 9 Output Saved.
PCA Mean Top-5 prediction time: 0.010567299999820534


### Step 10: Building the Top-10 Peer Space Using the 75% Variance PCA Latent Representation

This code constructs the Top-10 peer reduced space for each target item using the PCA latent space where the number of components k is chosen dynamically based on the 75% explained variance rule (k = k_75). It creates an item embedding matrix (items × k) from the top eigenvectors, then computes cosine similarity between each target item and all other items in this latent space. For each target item, the code selects the 10 most similar peer items, stores their ranks and similarity weights, and saves the full peer-space table to a CSV file for later recommendation and prediction steps.

As an extra verification step, the code also generates user reduced vectors by extracting each target user’s mean-centered ratings on the Top-10 peer items for every target item. These vectors provide a compact representation of user preferences within the Top-10 latent peer space and are saved to a separate CSV file for evaluation and TA-proof documentation.

In [153]:
k = k_75

latent_item_features_10 = eigenvectors[:, :k]
latent_item_df_10 = pd.DataFrame(latent_item_features_10, index=user_item_matrix_centered.columns)

reduced_space_data_10 = []
peers_dict_10 = {}

for tid in target_item_ids:
    target_vec = latent_item_df_10.loc[tid].values.reshape(1, -1)
    sim_scores = cosine_similarity(target_vec, latent_item_df_10.values).flatten()
    sim_series = pd.Series(sim_scores, index=latent_item_df_10.index).drop(tid)
    sorted_peers = sim_series.sort_values(ascending=False)

    top10 = sorted_peers.head(10).index.tolist()

    peers_dict_10[tid] = {
        'top10': top10,
        'sim_series': sim_series
    }

    for rank, peer_id in enumerate(top10, 1):
        weight = sim_series[peer_id]
        reduced_space_data_10.append({
            'TargetItem': tid,
            'Peer_Rank': rank,
            'Peer_ItemID': peer_id,
            'Latent_Similarity': weight,
            'Space_Type': 'Top10_Latent'
        })

df_reduced_10 = pd.DataFrame(reduced_space_data_10)
print(df_reduced_10.head())

df_reduced_10.to_csv(tables_path + 'pca_step10_reduced_space_top10.csv', index=False)
print("Step 10 Output Saved: pca_step10_reduced_space_top10.csv")

rows = []
for tid in target_item_ids:
    peers = peers_dict_10[tid]['top10']
    for uid in target_user_ids:
        if uid not in user_item_matrix_centered.index:
            continue

        vec = user_item_matrix_centered.loc[uid, peers].values 
        row = {'UserID': uid, 'TargetItem': tid}
        for i, p in enumerate(peers, 1):
            row[f'Peer{i}_{p}'] = vec[i-1]
        rows.append(row)

df_user_red10 = pd.DataFrame(rows)
print(df_user_red10.head())

df_user_red10.to_csv(tables_path + "pca_step10_user_reduced_vectors_top10.csv", index=False)
print("Step 10 (extra) Output Saved: pca_step10_user_reduced_vectors_top10.csv")

   TargetItem  Peer_Rank  Peer_ItemID  Latent_Similarity    Space_Type
0       99904          1           34                0.0  Top10_Latent
1       99904          2        92210                0.0  Top10_Latent
2       99904          3        94786                0.0  Top10_Latent
3       99904          4        94739                0.0  Top10_Latent
4       99904          5        94405                0.0  Top10_Latent


Step 10 Output Saved: pca_step10_reduced_space_top10.csv
   UserID  TargetItem      Peer1_34   Peer2_92210  Peer3_94786   Peer4_94739  \
0       1       99904  8.881784e-16  8.881784e-16          0.0  1.332268e-15   
1     134       99904  2.833118e-01  8.881784e-16          0.0  1.332268e-15   
2     903       99904  2.833118e-01  8.881784e-16          0.0  1.332268e-15   
3       1      119705  8.881784e-16  8.881784e-16          0.0  1.332268e-15   
4     134      119705  2.833118e-01  8.881784e-16          0.0  1.332268e-15   

   Peer5_94405  Peer6_93752  Peer7_93574  Peer8_93570  Peer9_93544  \
0          0.0          0.0          0.0          0.0          0.0   
1          0.0          0.0          0.0          0.0          0.0   
2          0.0          0.0          0.0          0.0          0.0   
3          0.0          0.0          0.0          0.0          0.0   
4          0.0          0.0          0.0          0.0          0.0   

   Peer10_93510  
0  8.881784e-16  
1  8.

### Step 11: Predicting Ratings Using Top-10 Latent Peers

This code generates rating predictions for each target user–item pair using the top 10 most similar peer items found in the PCA latent space. For every target user and target item, it first checks whether the rating already exists in the original user–item matrix or is missing, and labels the case accordingly.

It then applies a similarity-weighted prediction formula using the user’s mean-centered ratings on the Top-10 peer items and their cosine similarity weights. If no valid similarity information is available (denominator equals zero), the prediction falls back to the item’s average rating. Otherwise, the weighted deviation is added back to the item mean to produce the final predicted rating. All predictions and status labels are stored in a DataFrame and saved to a CSV file for evaluation and analysis.

In [154]:
t0 = time.perf_counter()

predictions_top10 = []

for uid in target_user_ids:
    if uid not in user_item_matrix.index: 
        continue

    for tid in target_item_ids:

        if pd.notna(user_item_matrix.loc[uid, tid]):
            status = "Existing"
        else:
            status = "Missing"

        current_peers = peers_dict_10[tid]['top10']
        sim_series = peers_dict_10[tid]['sim_series']

        numerator = 0
        denominator = 0

        for peer_id in current_peers:
            weight = sim_series[peer_id]
            val_centered = user_item_matrix_centered.loc[uid, peer_id]
            numerator += weight * val_centered
            denominator += abs(weight)

        if denominator == 0:
            pred = item_means[tid]
        else:
            pred = item_means[tid] + (numerator / denominator)

        predictions_top10.append({'UserID': uid, 'ItemID': tid, 'Pred_Top10': pred, 'Status': status})

df_pred10 = pd.DataFrame(predictions_top10)
print(df_pred10)
df_pred10.to_csv(tables_path + 'pca_step11_predictions_top10.csv', index=False)
print("Step 11 Output Saved.")

t1 = time.perf_counter()
log_time("PCA_MeanFill", "Prediction_Top10", t1 - t0)
print("PCA Mean Top-10 prediction time:", t1 - t0)

   UserID  ItemID  Pred_Top10   Status
0       1   99904         1.0  Missing
1       1  119705         1.0  Missing
2     134   99904         1.0  Missing
3     134  119705         1.0  Missing
4     903   99904         1.0  Missing
5     903  119705         1.0  Missing
Step 11 Output Saved.
PCA Mean Top-10 prediction time: 0.010082500000862638


In [155]:
pd.DataFrame(runtime_rows).to_csv(tables_path + "pca_meanfill_runtime.csv", index=False)

### Step 12: Comparing Top-5 and Top-10 Latent Space Predictions

This code compares the rating predictions generated using the Top-5 and Top-10 latent peer models. It merges the prediction results from Step 9 and Step 11 based on user and item identifiers, ensuring that both predictions are aligned for each user–item pair. The code then computes the difference and absolute difference between the two prediction methods to quantify how much the predicted ratings change when using more peers in the latent space.

The merged comparison results are printed and saved to a CSV file for analysis. Finally, the code calculates and reports the average absolute difference, providing a concise numerical summary of the impact of increasing the latent space neighborhood from 5 to 10 peers.

In [156]:
merged = pd.merge(df_pred5[['UserID', 'ItemID', 'Pred_Top5', 'Status']], 
                  df_pred10[['UserID', 'ItemID', 'Pred_Top10']], 
                  on=['UserID', 'ItemID'])

merged['Diff'] = merged['Pred_Top10'] - merged['Pred_Top5']
merged['AbsDiff'] = merged['Diff'].abs()

print("Comparison of Top 5 vs Top 10 Predictions:")
print(merged)

merged.to_csv(tables_path + 'pca_step12_comparison.csv', index=False)
print("Step 12 Output Saved.")

avg_diff = merged['AbsDiff'].mean()
print(f"\nAverage Absolute Difference: {avg_diff:.4f}")

Comparison of Top 5 vs Top 10 Predictions:
   UserID  ItemID  Pred_Top5   Status  Pred_Top10  Diff  AbsDiff
0       1   99904        1.0  Missing         1.0   0.0      0.0
1       1  119705        1.0  Missing         1.0   0.0      0.0
2     134   99904        1.0  Missing         1.0   0.0      0.0
3     134  119705        1.0  Missing         1.0   0.0      0.0
4     903   99904        1.0  Missing         1.0   0.0      0.0
5     903  119705        1.0  Missing         1.0   0.0      0.0
Step 12 Output Saved.

Average Absolute Difference: 0.0000
